# ATOM3D-LBA Pocket Graph Regression with GNNVisualizer

This notebook trains a graph-level GraphSAGE regressor on the Hugging Face `vector-institute/atom3d-lba` dataset, then renders a cropped protein-ligand pocket graph with `GNNVisualizer`.

The task is ligand binding affinity regression: predict pK from a co-crystallized protein-ligand complex. The Hugging Face card exposes atomic numbers, 3D coordinates, pK labels, and token-type masks for protein, pocket, and ligand atoms. For visualization, the notebook keeps ligand atoms plus nearby pocket/protein atoms so the displayed graph stays around the 300-500 node range.

Source docs: [ATOM3D](https://www.atom3d.ai/), [ATOM3D-LBA Hugging Face dataset](https://huggingface.co/datasets/vector-institute/atom3d-lba), and the [GVP-GNN ATOM3D benchmark summary](https://icml-compbio.github.io/icml-website-2021/2021/papers/WCBICML2021_paper_15.pdf).

If imports fail in a fresh kernel, install the runtime packages first:

```bash
python3 -m pip install torch torch-geometric datasets
```

Optional environment variables: `ATOM3D_LBA_EPOCHS`, `ATOM3D_LBA_MAX_GRAPHS`, `ATOM3D_LBA_TARGET_NODES`, and `ATOM3D_LBA_HIDDEN_CHANNELS`.

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from IPython.display import Markdown, display
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool

from gnn_exp import GNNVisualizer

In [ ]:
SEED = 7
torch.manual_seed(SEED)

EPOCHS = int(os.environ.get("ATOM3D_LBA_EPOCHS", "3"))
MAX_GRAPHS = int(os.environ.get("ATOM3D_LBA_MAX_GRAPHS", "64"))
TARGET_NODES = int(os.environ.get("ATOM3D_LBA_TARGET_NODES", "420"))
HIDDEN_CHANNELS = int(os.environ.get("ATOM3D_LBA_HIDDEN_CHANNELS", "16"))
BATCH_SIZE = 8
EDGE_RADIUS = float(os.environ.get("ATOM3D_LBA_EDGE_RADIUS", "4.5"))
KNN_EDGES = int(os.environ.get("ATOM3D_LBA_KNN_EDGES", "8"))


def nearest_to_ligand(indices, coords, ligand_coords, limit):
    if limit <= 0 or indices.numel() == 0:
        return indices[:0]
    distances = torch.cdist(coords[indices], ligand_coords).min(dim=1).values
    order = distances.argsort()
    return indices[order[:limit]]


def crop_atom_indices(coords, token_type, target_nodes):
    ligand_idx = (token_type == 2).nonzero(as_tuple=False).view(-1)
    pocket_idx = (token_type == 1).nonzero(as_tuple=False).view(-1)
    protein_idx = (token_type == 0).nonzero(as_tuple=False).view(-1)
    if ligand_idx.numel() == 0:
        ligand_idx = torch.arange(min(coords.size(0), max(1, target_nodes // 10)))
    ligand_coords = coords[ligand_idx]
    if ligand_idx.numel() > target_nodes:
        center = ligand_coords.mean(dim=0, keepdim=True)
        distances = torch.cdist(ligand_coords, center).view(-1)
        return ligand_idx[distances.argsort()[:target_nodes]]
    remaining = target_nodes - ligand_idx.numel()
    pocket_keep = nearest_to_ligand(pocket_idx, coords, ligand_coords, remaining)
    remaining -= pocket_keep.numel()
    protein_keep = nearest_to_ligand(protein_idx, coords, ligand_coords, remaining)
    keep = torch.cat([ligand_idx, pocket_keep, protein_keep]).unique(sorted=True)
    return keep


def build_edges(coords, radius=EDGE_RADIUS, k=KNN_EDGES):
    distances = torch.cdist(coords, coords)
    radius_mask = (distances <= radius) & (distances > 0)
    radius_edges = radius_mask.nonzero(as_tuple=False).t().contiguous()
    k = min(max(1, k), max(coords.size(0) - 1, 1))
    nearest = distances.topk(k + 1, largest=False).indices[:, 1:]
    sources = torch.arange(coords.size(0)).view(-1, 1).expand_as(nearest).reshape(-1)
    knn_edges = torch.stack([sources, nearest.reshape(-1)], dim=0)
    edge_index = torch.cat([radius_edges, knn_edges], dim=1)
    edge_index = torch.unique(edge_index, dim=1)
    reverse_edges = edge_index.flip(0)
    return torch.unique(torch.cat([edge_index, reverse_edges], dim=1), dim=1)


def atom_features(input_ids, token_type, coords):
    centered = coords - coords.mean(dim=0, keepdim=True)
    scaled_coords = centered / centered.std(dim=0, keepdim=True).clamp_min(1.0)
    atom_number = input_ids.float().view(-1, 1) / input_ids.float().max().clamp_min(1.0)
    token_features = F.one_hot(token_type.clamp(0, 2), num_classes=3).float()
    ligand_coords = coords[token_type == 2]
    center = ligand_coords.mean(dim=0, keepdim=True) if ligand_coords.numel() else coords.mean(dim=0, keepdim=True)
    ligand_distance = torch.cdist(coords, center).view(-1, 1)
    ligand_distance = ligand_distance / ligand_distance.max().clamp_min(1.0)
    return torch.cat([atom_number, token_features, scaled_coords, ligand_distance], dim=1)


def make_graph(example, target_nodes=TARGET_NODES):
    input_ids = torch.tensor(example["input_ids"], dtype=torch.long)
    coords = torch.tensor(example["coords"], dtype=torch.float32)
    token_type = torch.tensor(example["token_type_ids"], dtype=torch.long)
    keep = crop_atom_indices(coords, token_type, target_nodes)
    input_ids = input_ids[keep]
    coords = coords[keep]
    token_type = token_type[keep]
    edge_index = build_edges(coords)
    return Data(
        x=atom_features(input_ids, token_type, coords),
        edge_index=edge_index,
        y=torch.tensor([float(example["labels"])], dtype=torch.float32),
        coords=coords,
        token_type=token_type,
    )


raw_dataset = load_dataset("vector-institute/atom3d-lba", split=f"train[:{MAX_GRAPHS}]")
graphs = [make_graph(raw_dataset[index]) for index in range(len(raw_dataset))]
label_values = torch.tensor([float(graph.y.item()) for graph in graphs])
target_mean = label_values.mean()
target_std = label_values.std(unbiased=False).clamp_min(1e-6)
train_cut = max(1, int(0.8 * len(graphs)))
train_graphs = graphs[:train_cut]
test_graphs = graphs[train_cut:] or graphs[:1]
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE)
visual_data = min(graphs, key=lambda data: abs(int(data.num_nodes) - TARGET_NODES))
query_pair = visual_data.edge_index[:, 0].tolist()

num_features = visual_data.x.size(1)

display(Markdown(
    f"Loaded **ATOM3D-LBA** sample with {len(graphs)} cropped pocket graphs. "
    f"Visual graph: {visual_data.num_nodes} nodes, {visual_data.edge_index.size(1)} directed edges, "
    f"{num_features} visible node features."
))

In [ ]:
class LBAGraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.act1 = nn.Tanh()
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.act2 = nn.Tanh()
        self.regressor = nn.Linear(hidden_channels, 1)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        x = self.act1(self.conv1(x, edge_index))
        x = self.act2(self.conv2(x, edge_index))
        graph_embedding = global_mean_pool(x, batch)
        return self.regressor(graph_embedding)

In [ ]:
def scaled_target(batch):
    return (batch.y.view(-1, 1).float() - target_mean) / target_std


def evaluate_mae(model, loader):
    model.eval()
    errors = []
    with torch.no_grad():
        for batch in loader:
            scaled_pred = model(batch.x, batch.edge_index, batch.batch)
            pred = scaled_pred * target_std + target_mean
            errors.append((pred.view(-1) - batch.y.view(-1).float()).abs())
    return float(torch.cat(errors).mean()) if errors else float("nan")


def train_model(model, loader, epochs=EPOCHS):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.006, weight_decay=1e-4)
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for batch in loader:
            optimizer.zero_grad()
            pred = model(batch.x, batch.edge_index, batch.batch)
            loss = F.mse_loss(pred, scaled_target(batch))
            loss.backward()
            optimizer.step()
            total_loss += float(loss.detach()) * batch.num_graphs
        if epoch == 1 or epoch == epochs:
            avg_loss = total_loss / max(len(loader.dataset), 1)
            display(Markdown(f"Epoch {epoch}: scaled train MSE {avg_loss:.4f}"))
    return model


model = LBAGraphSAGE(num_features, HIDDEN_CHANNELS)
model = train_model(model, train_loader)
mae = evaluate_mae(model, test_loader)
display(Markdown(f"Held-out MAE on the small demo split: **{mae:.3f} pK**"))

The next cell builds the widget for the cropped pocket graph. `renderer="auto"` keeps the visualization on WebGPU/WebGL when available.

In [ ]:
visualizer = GNNVisualizer(viewportHeight=1120, autoFit=True)
visualizer.add_model(
    data=visual_data,
    model=model.eval(),
    subgraphSample=False,
    queries=[query_pair],
    mode="graph",
)

assert visualizer.renderer == "auto"
assert visualizer.autoFit is True
assert visualizer.viewportHeight == 1120
assert visualizer.modelInfo["conv1"]["type"] == "SAGEConv"
assert visualizer.modelInfo["conv1"].get("aggregation") == "mean"
assert len(visualizer.graphData["x"]) == visual_data.num_nodes
assert "graphAggregation" in visualizer.intmData
assert len(visualizer.intmData["act1"][0]) == HIDDEN_CHANNELS

display(Markdown(
    "| Captured object | Value |\n"
    "| --- | ---: |\n"
    f"| Nodes | {len(visualizer.graphData['x'])} |\n"
    f"| Edges | {visual_data.edge_index.size(1)} |\n"
    f"| Hidden channels | {HIDDEN_CHANNELS} |\n"
    f"| Viewport height | {visualizer.viewportHeight}px |"
))

In [ ]:
display(visualizer)